# Spark Practice — Data Sources API with real data
**Dataset:** `games.csv` from *Football Data from Transfermarkt* (Kaggle: davidcariboo/player-scores).
Each row is a match: competition, season, date, teams, goals, attendance, stadium…

**Goal:** repeat Session 12 with my own data: CSV (inferSchema vs explicit schema), JSON, Parquet and partitioning.

## 0. Start Spark
`SparkSession` is the entry point. When it is created, Python launches a **Java (JVM)** process under the hood — that is the real Spark engine. My Python code only sends it instructions.
- `master("local[*]")` → use all the cores of my machine as if they were a mini-cluster.
- `RUTA` / `PATH` → the CSV is in the same folder as the notebook.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
import os, time

spark = SparkSession.builder.master("local[*]").appName("PracticeGames").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

PATH = "games.csv"
print("Does the file exist?", os.path.exists(PATH))

## 1. Read the CSV letting Spark guess the types (`inferSchema`)
- `header=True` → the first row contains the column names.
- `inferSchema=True` → Spark makes an **extra pass** over the data to guess each column's type. Convenient, but slower and it can be wrong.
- `escape='"'` → some text fields (stadiums, names) contain quotes or commas; this prevents columns from breaking.

👀 Look at the `aggregate` column (results like `"2:1"`): Spark infers it as **timestamp**, because it reads `2:1` as a time of day. A classic `inferSchema` mistake.

In [ ]:
df_inf = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .option("escape", '"')
          .csv(PATH))

df_inf.printSchema()

df_inf.select("game_id", "competition_id", "season", "date",
              "home_club_name", "away_club_name",
              "home_club_goals", "away_club_goals", "attendance").show(5, truncate=False)

print("Rows:", df_inf.count(), "| Columns:", len(df_inf.columns))

## 2. Read with an explicit schema
I tell Spark the type of each column with a `StructType`:
- No extra inference pass → faster on large files.
- `aggregate` becomes a **string** (with inferSchema it was wrongly a timestamp).
- If a value doesn't fit its type, Spark sets it to `null` instead of failing (`PERMISSIVE` mode, the default).

⚠️ In CSV the schema is applied **by position, not by name**: the columns must be in the same order as in the file — and the real order of `games.csv` does **not** match the Kaggle documentation.

In [ ]:
schema = T.StructType([
    T.StructField("game_id",                T.IntegerType()),
    T.StructField("competition_id",         T.StringType()),
    T.StructField("season",                 T.IntegerType()),
    T.StructField("round",                  T.StringType()),
    T.StructField("date",                   T.DateType()),
    T.StructField("home_club_id",           T.IntegerType()),
    T.StructField("away_club_id",           T.IntegerType()),
    T.StructField("home_club_goals",        T.IntegerType()),
    T.StructField("away_club_goals",        T.IntegerType()),
    T.StructField("home_club_position",     T.IntegerType()),
    T.StructField("away_club_position",     T.IntegerType()),
    T.StructField("home_club_manager_name", T.StringType()),
    T.StructField("away_club_manager_name", T.StringType()),
    T.StructField("stadium",                T.StringType()),
    T.StructField("attendance",             T.IntegerType()),
    T.StructField("referee",                T.StringType()),
    T.StructField("url",                    T.StringType()),
    T.StructField("home_club_formation",    T.StringType()),
    T.StructField("away_club_formation",    T.StringType()),
    T.StructField("home_club_name",         T.StringType()),
    T.StructField("away_club_name",         T.StringType()),
    T.StructField("aggregate",              T.StringType()),   # "2:1" -> text, NOT timestamp
    T.StructField("competition_type",       T.StringType()),
])

games = (spark.read
         .option("header", True)
         .option("escape", '"')
         .schema(schema)
         .csv(PATH))

games.printSchema()

### Check: does my schema match the real column order of the file?
I compare, position by position, the names in my schema with the CSV header (the columns of `df_inf`).

In [ ]:
wrong = [(i, mine, real) for i, (mine, real) in enumerate(zip(games.columns, df_inf.columns)) if mine != real]
print("Misplaced columns:", wrong if wrong else "none ✅")

games.select("game_id", "date", "home_club_name", "away_club_name",
             "home_club_goals", "away_club_goals", "aggregate", "stadium").show(5, truncate=False)

## 3. Data quality: how many nulls?
For each column I turn `isNull()` into 0/1 and sum it → number of nulls per column, all computed in **a single pass** over the data (one `select` with one expression per column).

In [ ]:
cols_to_check = ["date", "season", "home_club_goals", "away_club_goals",
                 "attendance", "home_club_position", "referee", "stadium"]

nulls = games.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols_to_check])
nulls.show()

## 4. Transformations → lazy evaluation
I create new columns and keep only the ones I need:
- `total_goals` = home + away goals
- `result` = `H` (home win), `A` (away win), `D` (draw)
- `month` = match month, taken from `date`
- `filter` → drop matches without recorded goals (not played)

Spark **executes nothing** here: it only records the "recipe". That's why it takes ~0 seconds.

In [ ]:
t0 = time.time()

games_clean = (games
    .withColumn("total_goals", F.col("home_club_goals") + F.col("away_club_goals"))
    .withColumn("result",
                F.when(F.col("home_club_goals") > F.col("away_club_goals"), "H")
                 .when(F.col("home_club_goals") < F.col("away_club_goals"), "A")
                 .otherwise("D"))
    .withColumn("month", F.month("date"))
    .filter(F.col("home_club_goals").isNotNull())
    .select("game_id", "competition_id", "competition_type", "season", "date", "month",
            "home_club_name", "away_club_name", "home_club_goals", "away_club_goals",
            "total_goals", "result", "attendance", "stadium"))

print(f"Transformations defined in {time.time() - t0:.3f} s")

## 5. Action → now it actually runs
`.show()` is an **action**: Spark takes the whole recipe (read CSV → new columns → filter → groupBy), optimizes it and executes it.
I compute, per competition, the number of matches and the average goals per match.
Useful codes: `ES1` LaLiga · `GB1` Premier League · `IT1` Serie A · `L1` Bundesliga · `FR1` Ligue 1 · `CL` Champions League.

In [ ]:
t0 = time.time()

goals = (games_clean
         .groupBy("competition_id")
         .agg(F.count("*").alias("matches"),
              F.round(F.avg("total_goals"), 2).alias("goals_per_match"))
         .orderBy(F.desc("matches")))

goals.show(10)
print(f"Action executed in {time.time() - t0:.2f} s")

## 6. Write and read JSON
I only save 1,000 rows (`limit(1000)`) so I can open the file and look at it.
- Spark writes a **folder**, not a file: inside there are one or more `part-xxxx.json` files (one per writing task) and a `_SUCCESS` marker.
- `mode("overwrite")`: if the folder already exists, it is replaced. Without it, a second run would fail (the default mode is `errorifexists`).

👀 When reading it back: `date` comes back as **string** and integers as **long** — JSON has no date type, so **type information is lost**. Columns also come back in alphabetical order.

In [ ]:
json_path = "outputs/games_json"

games_clean.limit(1000).write.mode("overwrite").json(json_path)

print("Folder contents:")
for f in sorted(os.listdir(json_path)):
    print("  ", f)

games_json = spark.read.json(json_path)
games_json.printSchema()
games_json.show(3, truncate=False)

## 7. Write and read Parquet, and compare it with the CSV
Parquet is a **columnar**, **compressed** format that **stores the schema inside the file**.
I write the full DataFrame and compare its size with the original CSV.

In [ ]:
parquet_path = "outputs/games_parquet"

games_clean.write.mode("overwrite").parquet(parquet_path)

def size_mb(path):
    if os.path.isfile(path):
        return os.path.getsize(path) / 1e6
    return sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(path) for f in fs) / 1e6

print(f"Original CSV : {size_mb(PATH):.1f} MB  (23 columns)")
print(f"Parquet      : {size_mb(parquet_path):.1f} MB  (14 columns)")

games_pq = spark.read.parquet(parquet_path)
games_pq.printSchema()
print("Rows:", games_pq.count())

## 8. Partition by season (`partitionBy`)
`partitionBy("season")` creates **one subfolder per season**: `season=2012/`, `season=2013/`…
Then I read it and filter `season == 2023`. In the plan (`explain`) I look for **`PartitionFilters`**: Spark **only opens that season's folder** and skips the rest without reading them → *partition pruning*.

**When to partition:** by columns that are often used in filters and have **few distinct values** (season, competition). **Never** by `game_id`: it would create ~89,000 folders with tiny files (the *small files* problem).

In [ ]:
part_path = "outputs/games_by_season"

games_clean.write.mode("overwrite").partitionBy("season").parquet(part_path)

folders = sorted(d for d in os.listdir(part_path) if d.startswith("season="))
print(f"{len(folders)} folders:", folders)

s23 = spark.read.parquet(part_path).filter(F.col("season") == 2023)
s23.explain(mode="formatted")
print("Matches in season 2023:", s23.count())

## 9. My question with Spark SQL: is there a home advantage in LaLiga?
I register the DataFrame (read from **Parquet**, as in a real project) as a **temporary view** and use plain SQL. Under the hood Spark generates **the same plan** as with the DataFrame API.
For each LaLiga season (`ES1`) I compute the % of home wins, draws and away wins.
`AVG(CASE WHEN ... THEN 1 ELSE 0 END)` → the mean of ones and zeros = a proportion.

In [ ]:
games_pq.createOrReplaceTempView("games")

home_advantage = spark.sql("""
    SELECT season,
           COUNT(*)                                               AS matches,
           ROUND(AVG(CASE WHEN result = 'H' THEN 1 ELSE 0 END), 3) AS pct_home,
           ROUND(AVG(CASE WHEN result = 'D' THEN 1 ELSE 0 END), 3) AS pct_draw,
           ROUND(AVG(CASE WHEN result = 'A' THEN 1 ELSE 0 END), 3) AS pct_away
    FROM games
    WHERE competition_id = 'ES1'
    GROUP BY season
    ORDER BY season
""")
home_advantage.show(20)

### 9b. Same question for the top 5 leagues: 2020/21 (no fans) vs the rest
With a single league (380 matches per season) natural variation is about ±2.5 points, so I repeat the analysis for the 5 big leagues.
- `isin(top5)` filters the five leagues.
- `pivot("period")` turns the values of `period` into columns → "2020 no fans" and "rest" side by side, like a pivot table in Excel.

In [ ]:
top5 = ["ES1", "GB1", "IT1", "L1", "FR1"]

comparison = (games_pq
    .filter(F.col("competition_id").isin(top5))
    .withColumn("period", F.when(F.col("season") == 2020, "2020 no fans").otherwise("rest"))
    .groupBy("competition_id")
    .pivot("period")
    .agg(F.round(F.avg((F.col("result") == "H").cast("int")), 3))
    .orderBy("competition_id"))

comparison.show()

## 10. Conclusions

**1. inferSchema vs explicit schema**
- With `inferSchema`, Spark read the `aggregate` column (results like `"2:1"`) as a **timestamp**: it interpreted the score as a time of day. With the explicit schema I define it as **string**, which is correct.
- `inferSchema` also forces Spark to make an **extra pass** over the data just to guess types; with an explicit schema that pass disappears.
- In CSV the schema is applied **by position, not by name**. The real order of `games.csv` did not match the Kaggle documentation, so I checked column by column that my schema matched the header (result: none misplaced).

**2. Data quality**
- 88,958 matches and 23 columns.
- Consistency check: LaLiga, Premier League and Serie A have 5,320 matches = 380 per season × 14 seasons; the Bundesliga, 4,284 = 306 × 14. No missing or extra matches.
- There are no matches without recorded goals (the `filter` removed no rows), but I keep it to protect the code against future dataset updates.

**3. Lazy evaluation**
- The transformations (`withColumn`, `filter`, `select`) were defined in ~0 seconds: Spark only builds the plan.
- The real computation happens with an **action** (`show`, `count`, `write`), which is why the time is concentrated in those cells.

**4. JSON**
- Spark writes a **folder** with `part-xxxx` files and a `_SUCCESS` marker, not a single file.
- When reading it back **types were lost**: `date` became `string` and integers became `long`, and columns came back in alphabetical order. JSON is flexible but does not preserve the schema.

**5. Parquet**
- **CSV: 29.0 MB → Parquet: 1.8 MB (~16× smaller)**, thanks to columnar storage and compression (values like `ES1` repeat thousands of times).
- Reading it back needed **no schema and no options**: `date` stayed `date` and goals stayed `integer`. **The schema travels inside the file.**
- Being columnar, if I query only 2 columns Spark reads only those 2 from disk; with CSV it would have to read entire rows.

**6. Partitioning**
- `partitionBy("season")` creates one folder per season (`season=2012/`, `season=2013/`…).
- When filtering `season == 2023`, the plan shows **`PartitionFilters`**: Spark only opens that season's folder and skips the rest (*partition pruning*).
- Partition by columns with **few distinct values** that are frequently filtered (season, competition), never by columns like `game_id`, which would create thousands of tiny files.

**7. Analysis: home advantage without fans (2020/21 season)**
- In LaLiga, 2020/21 has the **lowest home-win rate of the series: 41.6%** vs 46.4% in the other seasons.
- With a single league this is not conclusive (with 380 matches per season natural variation is ±2.5 points), so I repeated the analysis for the top 5 leagues with `pivot`:

| League | 2020 no fans | Rest | Drop |
|---|---|---|---|
| Ligue 1 | 37.4% | 44.7% | −7.3 pts |
| Premier League | 37.9% | 45.1% | −7.2 pts |
| LaLiga | 41.6% | 46.4% | −4.8 pts |
| Bundesliga | 42.2% | 44.7% | −2.5 pts |
| Serie A | 41.1% | 43.0% | −1.9 pts |

- **Home advantage drops in all five leagues.** Together they add up to ~1,800 matches in 2020 (natural variation ±1.2 pts) and the average drop is ~5 points: the effect is clear and not explained by chance.
- **Caveat:** 2020/21 also had a compressed calendar and 5 substitutions per match, so the result supports that fans matter but does not isolate that cause on its own.
- Underlying trend in LaLiga: home advantage decreases over the years (~49% in 2012 → ~44% in 2023-24) while draws increase (22% → 28%).

**8. What would change if `games.csv` were 100 GB?**
- The code would be practically the same: only the `master` would change (a cluster instead of `local[*]`).
- What is "good practice" here would become essential: an explicit schema (avoid the inference pass), converting to Parquet once and always working on it, and partitioning by the most frequently filtered columns.